# Data Cleaning and Validation
## 1. Importing the data

In [1]:
import pandas as pd

plant2_weather = pd.read_csv('C:/Users/acer/programming/jupyter/2026/solar_power_generation_data/data/Plant_2_Weather_Sensor_Data (1).csv', sep=",")

print(f"Plant 2 Weather Sensor: {plant2_weather.shape}")


plant2_weather.head()

Plant 2 Weather Sensor: (3259, 6)


,DATE_TIME,PLANT_ID,SOURCE_KEY,AMBIENT_TEMPERATURE,MODULE_TEMPERATURE,IRRADIATION
0,2020-05-15 00:00:00,4136001,iq8k7ZNt4Mwm3w0,27.004764,25.060789,0.0
1,2020-05-15 00:15:00,4136001,iq8k7ZNt4Mwm3w0,26.880811,24.421869,0.0
2,2020-05-15 00:30:00,4136001,iq8k7ZNt4Mwm3w0,26.682055,24.427290,0.0
3,2020-05-15 00:45:00,4136001,iq8k7ZNt4Mwm3w0,26.500589,24.420678,0.0
4,2020-05-15 01:00:00,4136001,iq8k7ZNt4Mwm3w0,26.596148,25.088210,0.0


# 2. Understanding the data
## 2.1. Looking at the data

In [2]:
print(plant2_weather.info())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3259 entries, 0 to 3258
Data columns (total 6 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   DATE_TIME            3259 non-null   object 
 1   PLANT_ID             3259 non-null   int64  
 2   SOURCE_KEY           3259 non-null   object 
 3   AMBIENT_TEMPERATURE  3259 non-null   float64
 4   MODULE_TEMPERATURE   3259 non-null   float64
 5   IRRADIATION          3259 non-null   float64
dtypes: float64(3), int64(1), object(2)
memory usage: 152.9+ KB
None


In [3]:
print(plant2_weather.describe())

        PLANT_ID  AMBIENT_TEMPERATURE  MODULE_TEMPERATURE  IRRADIATION
count     3259.0          3259.000000         3259.000000  3259.000000
mean   4136001.0            28.069400           32.772408     0.232737
std          0.0             4.061556           11.344034     0.312693
min    4136001.0            20.942385           20.265123     0.000000
25%    4136001.0            24.602135           23.716881     0.000000
50%    4136001.0            26.981263           27.534606     0.019040
75%    4136001.0            31.056757           40.480653     0.438717
max    4136001.0            39.181638           66.635953     1.098766


## 2.2 Defining Column Constraints and Data Types

In [4]:
# ==========================================
# STEP 2: DEFINE CONSTRAINTS FOR EACH COLUMN
# ==========================================

CONSTRAINTS = {
    'DATE_TIME': {
        'description': 'Timestamp of measurement',
        'expected_type': 'datetime',
        'format_variants': ['%Y-%m-%d %H:%M:%S'],
        'range': ('2020-05-15 00:00:00', '2020-06-17 23:45:00'),
        'nullable': False
    },
    'PLANT_ID': {
        'description': 'Solar plant identifier',
        'expected_type': 'integer',
        'allowed_values': [4136001],
        'nullable': False
    },
    'SOURCE_KEY': {
        'description': 'Weather sensor identifier',
        'expected_type': 'string',
        'nullable': False
    },
    'AMBIENT_TEMPERATURE': {
        'description': 'Ambient temperature (°C)',
        'expected_type': 'float',
        'min': -50.0,
        'max': 60.0,
        'nullable': False,
        'decimal_separator': '.'
    },
    'MODULE_TEMPERATURE': {
        'description': 'Solar module temperature (°C)',
        'expected_type': 'float',
        'min': -50.0,
        'max': 85.0,
        'nullable': False,
        'decimal_separator': '.'
    },
    'IRRADIATION': {
        'description': 'Solar irradiance (kW/m²)',
        'expected_type': 'float',
        'min': 0.0,
        'max': 1.5,
        'nullable': False,
        'decimal_separator': '.'
    }
}

print("=== CONSTRAINTS DEFINED ===")
for col, rules in CONSTRAINTS.items():
    print(f"\n{col}: {rules['description']}")
    for rule, value in rules.items():
        if rule != 'description':
            print(f"  - {rule}: {value}")

=== CONSTRAINTS DEFINED ===

DATE_TIME: Timestamp of measurement
  - expected_type: datetime
  - format_variants: ['%Y-%m-%d %H:%M:%S']
  - range: ('2020-05-15 00:00:00', '2020-06-17 23:45:00')
  - nullable: False

PLANT_ID: Solar plant identifier
  - expected_type: integer
  - allowed_values: [4136001]
  - nullable: False

SOURCE_KEY: Weather sensor identifier
  - expected_type: string
  - nullable: False

AMBIENT_TEMPERATURE: Ambient temperature (°C)
  - expected_type: float
  - min: -50.0
  - max: 60.0
  - nullable: False
  - decimal_separator: .

MODULE_TEMPERATURE: Solar module temperature (°C)
  - expected_type: float
  - min: -50.0
  - max: 85.0
  - nullable: False
  - decimal_separator: .

IRRADIATION: Solar irradiance (kW/m²)
  - expected_type: float
  - min: 0.0
  - max: 1.5
  - nullable: False
  - decimal_separator: .


## 3. Data Cleaning
#### 3.1. Check for missing values

In [5]:
# Check for missing values
missing_values = plant2_weather.isnull().sum()
print("Missing values per column:")
print(missing_values)
print("\nPercentage missing:")
print((missing_values / len(plant2_weather)) * 100)

Missing values per column:
DATE_TIME              0
PLANT_ID               0
SOURCE_KEY             0
AMBIENT_TEMPERATURE    0
MODULE_TEMPERATURE     0
IRRADIATION            0
dtype: int64

Percentage missing:
DATE_TIME              0.0
PLANT_ID               0.0
SOURCE_KEY             0.0
AMBIENT_TEMPERATURE    0.0
MODULE_TEMPERATURE     0.0
IRRADIATION            0.0
dtype: float64


#### 3.2. Check for duplicate values


In [6]:
# Check for duplicate rows
duplicate_rows = plant2_weather.duplicated().sum()
print(f"Number of duplicate rows: {duplicate_rows}")

if duplicate_rows > 0:
    print("\nDuplicate rows (first 5):")
    print(plant2_weather[plant2_weather.duplicated(keep=False)].head())

Number of duplicate rows: 0


#### 3.3. Check for outliers


In [7]:
# Check for outliers using IQR method
import numpy as np

numeric_columns = ['AMBIENT_TEMPERATURE', 'MODULE_TEMPERATURE', 'IRRADIATION']

for col in numeric_columns:
    Q1 = plant2_weather[col].quantile(0.25)
    Q3 = plant2_weather[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = plant2_weather[(plant2_weather[col] < lower_bound) | (plant2_weather[col] > upper_bound)]
    print(f"\n{col}:")
    print(f"  Lower bound: {lower_bound:.2f}")
    print(f"  Upper bound: {upper_bound:.2f}")
    print(f"  Number of outliers: {len(outliers)}")
    if len(outliers) > 0:
        print(f"  Outlier range: {outliers[col].min():.2f} to {outliers[col].max():.2f}")


AMBIENT_TEMPERATURE:
  Lower bound: 14.92
  Upper bound: 40.74
  Number of outliers: 0

MODULE_TEMPERATURE:
  Lower bound: -1.43
  Upper bound: 65.63
  Number of outliers: 2
  Outlier range: 66.02 to 66.64

IRRADIATION:
  Lower bound: -0.66
  Upper bound: 1.10
  Number of outliers: 1
  Outlier range: 1.10 to 1.10


#### 3.4. Check for consistency in data


In [8]:
# Check for data consistency
print("=== DATA CONSISTENCY CHECKS ===")

# Check if MODULE_TEMPERATURE is always >= AMBIENT_TEMPERATURE (should be true as modules heat up in sun)
module_colder = plant2_weather[plant2_weather['MODULE_TEMPERATURE'] < plant2_weather['AMBIENT_TEMPERATURE']]
print(f"\nMODULE_TEMPERATURE < AMBIENT_TEMPERATURE: {len(module_colder)} records")
if len(module_colder) > 0:
    print("Sample records where module is colder than ambient:")
    print(module_colder[['DATE_TIME', 'AMBIENT_TEMPERATURE', 'MODULE_TEMPERATURE']].head())

# Check for impossible irradiation values during night hours
plant2_weather['DATE_TIME_DT'] = pd.to_datetime(plant2_weather['DATE_TIME'], format='%Y-%m-%d %H:%M:%S')
plant2_weather['HOUR'] = plant2_weather['DATE_TIME_DT'].dt.hour
night_irradiation = plant2_weather[(plant2_weather['HOUR'] >= 20) | (plant2_weather['HOUR'] <= 5)]
night_irradiation_positive = night_irradiation[night_irradiation['IRRADIATION'] > 0]
print(f"\nNight hours (20:00-05:00) with positive irradiation: {len(night_irradiation_positive)} records")

# Check for reasonable temperature differences
temp_diff = plant2_weather['MODULE_TEMPERATURE'] - plant2_weather['AMBIENT_TEMPERATURE']
extreme_diff = plant2_weather[(temp_diff > 30) | (temp_diff < -10)]
print(f"\nExtreme temperature differences (>30°C or <-10°C): {len(extreme_diff)} records")
if len(extreme_diff) > 0:
    print(f"Temperature difference range: {temp_diff.min():.2f} to {temp_diff.max():.2f}")

# Clean up temporary columns
plant2_weather.drop(['DATE_TIME_DT', 'HOUR'], axis=1, inplace=True, errors='ignore')

=== DATA CONSISTENCY CHECKS ===

MODULE_TEMPERATURE < AMBIENT_TEMPERATURE: 1717 records
Sample records where module is colder than ambient:
             DATE_TIME  AMBIENT_TEMPERATURE  MODULE_TEMPERATURE
0  2020-05-15 00:00:00            27.004764           25.060789
1  2020-05-15 00:15:00            26.880811           24.421869
2  2020-05-15 00:30:00            26.682055           24.427290
3  2020-05-15 00:45:00            26.500589           24.420678
4  2020-05-15 01:00:00            26.596148           25.088210

Night hours (20:00-05:00) with positive irradiation: 86 records

Extreme temperature differences (>30°C or <-10°C): 2 records
Temperature difference range: -3.98 to 33.45


#### 3.5. Check for data type consistency

In [9]:
# Check data type consistency
print("=== DATA TYPE CONSISTENCY CHECKS ===")

# Expected types based on constraints
expected_types = {
    'DATE_TIME': 'object',  # Will be converted to datetime
    'PLANT_ID': 'int64',
    'SOURCE_KEY': 'object',
    'AMBIENT_TEMPERATURE': 'float64',
    'MODULE_TEMPERATURE': 'float64',
    'IRRADIATION': 'float64'
}

print("\nCurrent data types:")
print(plant2_weather.dtypes)

print("\nData type validation:")
type_issues = []
for col, expected_type in expected_types.items():
    actual_type = str(plant2_weather[col].dtype)
    if actual_type != expected_type:
        type_issues.append(f"{col}: expected {expected_type}, got {actual_type}")
        print(f"  ❌ {col}: expected {expected_type}, got {actual_type}")
    else:
        print(f"  ✅ {col}: {actual_type}")

if type_issues:
    print(f"\n⚠️  Found {len(type_issues)} data type issues")
else:
    print("\n✅ All data types are as expected")

# Test DATE_TIME conversion
print("\nTesting DATE_TIME format conversion...")
try:
    test_dates = pd.to_datetime(plant2_weather['DATE_TIME'].head(10), format='%Y-%m-%d %H:%M:%S')
    print("✅ DATE_TIME format is valid")
    print(f"Sample converted dates: {test_dates.tolist()[:3]}")
except Exception as e:
    print(f"❌ DATE_TIME format error: {e}")

=== DATA TYPE CONSISTENCY CHECKS ===

Current data types:
DATE_TIME               object
PLANT_ID                 int64
SOURCE_KEY              object
AMBIENT_TEMPERATURE    float64
MODULE_TEMPERATURE     float64
IRRADIATION            float64
dtype: object

Data type validation:
  ✅ DATE_TIME: object
  ✅ PLANT_ID: int64
  ✅ SOURCE_KEY: object
  ✅ AMBIENT_TEMPERATURE: float64
  ✅ MODULE_TEMPERATURE: float64
  ✅ IRRADIATION: float64

✅ All data types are as expected

Testing DATE_TIME format conversion...
✅ DATE_TIME format is valid
Sample converted dates: [Timestamp('2020-05-15 00:00:00'), Timestamp('2020-05-15 00:15:00'), Timestamp('2020-05-15 00:30:00')]


#### 3.6. Check for plausibility of data


In [10]:
# Check data plausibility
print("=== DATA PLAUSIBILITY CHECKS ===")

# Check against defined constraints
constraint_violations = []

# 1. Check PLANT_ID values
invalid_plant_ids = plant2_weather[~plant2_weather['PLANT_ID'].isin(CONSTRAINTS['PLANT_ID']['allowed_values'])]
if len(invalid_plant_ids) > 0:
    constraint_violations.append(f"PLANT_ID: {len(invalid_plant_ids)} invalid values")

# 2. Check numeric ranges
for col in ['AMBIENT_TEMPERATURE', 'MODULE_TEMPERATURE', 'IRRADIATION']:
    min_val = CONSTRAINTS[col]['min']
    max_val = CONSTRAINTS[col]['max']

    below_min = plant2_weather[plant2_weather[col] < min_val]
    above_max = plant2_weather[plant2_weather[col] > max_val]

    if len(below_min) > 0:
        constraint_violations.append(f"{col}: {len(below_min)} values below minimum ({min_val})")

    if len(above_max) > 0:
        constraint_violations.append(f"{col}: {len(above_max)} values above maximum ({max_val})")

# 3. Check date range
try:
    dates = pd.to_datetime(plant2_weather['DATE_TIME'], format='%Y-%m-%d %H:%M:%S')
    min_date = pd.to_datetime(CONSTRAINTS['DATE_TIME']['range'][0], format='%Y-%m-%d %H:%M:%S')
    max_date = pd.to_datetime(CONSTRAINTS['DATE_TIME']['range'][1], format='%Y-%m-%d %H:%M:%S')

    invalid_dates = dates[(dates < min_date) | (dates > max_date)]
    if len(invalid_dates) > 0:
        constraint_violations.append(f"DATE_TIME: {len(invalid_dates)} values outside expected range")
except Exception as e:
    constraint_violations.append(f"DATE_TIME: Format conversion error - {e}")

# 4. Business logic checks
# Check for physically impossible temperature values
extreme_ambient = plant2_weather[(plant2_weather['AMBIENT_TEMPERATURE'] < -40) | (plant2_weather['AMBIENT_TEMPERATURE'] > 50)]
extreme_module = plant2_weather[(plant2_weather['MODULE_TEMPERATURE'] < -40) | (plant2_weather['MODULE_TEMPERATURE'] > 80)]

if len(extreme_ambient) > 0:
    constraint_violations.append(f"AMBIENT_TEMPERATURE: {len(extreme_ambient)} extreme values")

if len(extreme_module) > 0:
    constraint_violations.append(f"MODULE_TEMPERATURE: {len(extreme_module)} extreme values")

# 5. Check for unusual irradiation patterns
# Irradiation should be 0 during night hours and positive during day
plant2_weather['DATE_TIME_DT'] = pd.to_datetime(plant2_weather['DATE_TIME'], format='%Y-%m-%d %H:%M:%S')
plant2_weather['HOUR'] = plant2_weather['DATE_TIME_DT'].dt.hour

# Check daytime (10:00-15:00) for zero irradiation
daytime_zero_irradiation = plant2_weather[(plant2_weather['HOUR'].between(10, 15)) & (plant2_weather['IRRADIATION'] == 0)]
if len(daytime_zero_irradiation) > 0:
    constraint_violations.append(f"IRRADIATION: {len(daytime_zero_irradiation)} zero values during peak daylight hours")

# Check for maximum irradiation values exceeding typical solar panel limits
max_irradiation = plant2_weather[plant2_weather['IRRADIATION'] > 1.2]
if len(max_irradiation) > 0:
    constraint_violations.append(f"IRRADIATION: {len(max_irradiation)} values exceeding typical maximum (1.2 kW/m²)")

print(f"\nFound {len(constraint_violations)} constraint violations:")
for violation in constraint_violations:
    print(f"  ⚠️  {violation}")

if len(constraint_violations) == 0:
    print("\n✅ All constraint checks passed!")
else:
    print(f"\n❌ Data quality issues detected. Review required.")

# Clean up temporary columns
plant2_weather.drop(['DATE_TIME_DT', 'HOUR'], axis=1, inplace=True, errors='ignore')

=== DATA PLAUSIBILITY CHECKS ===

Found 0 constraint violations:

✅ All constraint checks passed!
